# TB-Guard: CWT Scalogram Training (Google Colab)

Notebook ini mengunduh dataset Kaggle `elffff/tuberculosis-cough-acoustics-curated-dataset`, mencari folder `Kaggle_RespiScan_Dataset/cwt_scalograms/{train,val,test}`, lalu melatih MobileNetV3-Small untuk klasifikasi gambar `Healthy` vs `TB`.

**Cara pakai**
1. Buka https://colab.research.google.com/
2. File → Upload notebook → unggah notebook ini.
3. Runtime → Change runtime type → pilih GPU jika tersedia.
4. Jalankan semua cell dari atas ke bawah.
5. Jika Kaggle meminta autentikasi, ikuti instruksi pada cell download. Jangan menempelkan token ke notebook atau membagikannya.

Progress bar tampil saat pemeriksaan data dan setiap batch training. Checkpoint disimpan setiap epoch agar training dapat dilanjutkan setelah sesi terputus jika Google Drive diaktifkan.

**Batasan:** sesi Colab bisa terputus atau mencapai batas penggunaan, jadi tidak ada notebook yang dapat menjamin runtime berjalan tanpa batas. Notebook ini mendukung resume. Model ini untuk eksperimen riset, bukan diagnosis klinis.

## 1. Instal dependensi

In [ ]:
%pip -q install -U kagglehub tqdm scikit-learn matplotlib pillow

## 2. Konfigurasi dan cek GPU

In [ ]:
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import os, re, time, random, json
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU tidak aktif; training bisa jauh lebih lambat.")

## 3. Unduh dataset melalui KaggleHub

Dataset publik biasanya dapat diunduh langsung. Jika terjadi error autentikasi/consent, buat Kaggle API token dan simpan di Colab Secrets dengan nama `KAGGLE_API_TOKEN`, aktifkan aksesnya untuk notebook, lalu jalankan ulang cell ini. Dokumentasi: https://github.com/Kaggle/kagglehub

In [ ]:
import kagglehub

DATASET_ID = "elffff/tuberculosis-cough-acoustics-curated-dataset"
try:
    print("Mengambil dataset (progress download akan ditampilkan oleh KaggleHub)...")
    dataset_path = kagglehub.dataset_download(DATASET_ID)
    print("Dataset path:", dataset_path)
except Exception:
    print("Download gagal. Jika error autentikasi/consent:")
    print("1. Kaggle → Settings → API → buat token.")
    print("2. Colab → Secrets (ikon kunci) → tambah KAGGLE_API_TOKEN.")
    print("3. Aktifkan akses notebook, lalu jalankan ulang cell ini.")
    raise

## 4. Cari folder scalogram dan hitung file

In [ ]:
SEARCH_ROOT = Path(dataset_path)
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def find_image_root(root):
    for p in root.rglob("cwt_scalograms"):
        if p.is_dir() and all((p / s).is_dir() for s in ["train", "val", "test"]):
            return p
    if all((root / s).is_dir() for s in ["train", "val", "test"]):
        return root
    return None

IMAGE_ROOT = find_image_root(SEARCH_ROOT)
if IMAGE_ROOT is None:
    print("Folder teratas yang tersedia:")
    for p in list(SEARCH_ROOT.iterdir())[:50]: print(p)
    raise FileNotFoundError("Tidak menemukan struktur cwt_scalograms/{train,val,test}.")

print("Image root:", IMAGE_ROOT)
for split in ["train", "val", "test"]:
    print("\n" + split.upper())
    for cls_dir in sorted(p for p in (IMAGE_ROOT / split).iterdir() if p.is_dir()):
        n = sum(1 for f in cls_dir.rglob("*") if f.suffix.lower() in IMAGE_EXTS)
        print(f"  {cls_dir.name}: {n:,} images")

## 5. Audit ID pasien antar-split

Ini adalah pemeriksaan heuristik berdasarkan nama file seperti `PID_97A`. Jika pasien yang sama ditemukan di lebih dari satu split, hasil evaluasi bisa terlalu optimistis. Pola nama file saja tidak menjamin independensi pasien.

In [ ]:
patient_splits = defaultdict(set)
unparsed = 0
for split in ["train", "val", "test"]:
    for f in (IMAGE_ROOT / split).rglob("*"):
        if f.suffix.lower() not in IMAGE_EXTS: continue
        m = re.search(r"(PID_[A-Za-z0-9]+)", f.name)
        if m: patient_splits[m.group(1)].add(split)
        else: unparsed += 1
overlaps = {pid: sorted(s) for pid, s in patient_splits.items() if len(s) > 1}
print("Patient IDs detected:", len(patient_splits))
print("IDs found in multiple splits:", len(overlaps))
print("Images without detected PID:", unparsed)
for pid, splits in list(overlaps.items())[:20]: print(pid, "->", splits)
if not overlaps: print("No overlap detected by the filename PID pattern.")

## 6. Buat DataLoader dan lihat contoh gambar

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder(IMAGE_ROOT / "train", transform=train_transform)
val_ds = datasets.ImageFolder(IMAGE_ROOT / "val", transform=eval_transform)
test_ds = datasets.ImageFolder(IMAGE_ROOT / "test", transform=eval_transform)
print("Classes:", train_ds.classes, "| mapping:", train_ds.class_to_idx)
print("Train/Val/Test:", len(train_ds), len(val_ds), len(test_ds))
assert train_ds.classes == val_ds.classes == test_ds.classes
assert "TB" in train_ds.class_to_idx and "Healthy" in train_ds.class_to_idx

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, cls in enumerate(["Healthy", "TB"]):
    paths = [p for p, y in train_ds.samples if train_ds.classes[y] == cls]
    for col, ax in enumerate(axes[row]):
        p = random.choice(paths)
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(cls); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Siapkan model CNN

MobileNetV3-Small relatif ringan. Notebook mencoba bobot pretrained ImageNet; jika unduhan bobot gagal, otomatis memakai inisialisasi acak. Pretrained weights biasanya membantu training, tetapi memerlukan akses unduhan.

In [ ]:
import torch.nn as nn
from torchvision import models

try:
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    print("Loaded pretrained ImageNet weights.")
except Exception as e:
    print("Pretrained weights unavailable; using random initialization.")
    print(str(e)[:250])
    model = models.mobilenet_v3_small(weights=None)

model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(train_ds.classes))
model = model.to(DEVICE)

USE_AMP = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
TB_IDX = train_ds.class_to_idx["TB"]
print("Model ready on", DEVICE)

## 8. Pilih lokasi checkpoint

Direkomendasikan menyimpan checkpoint ke Google Drive agar bisa dilanjutkan jika runtime Colab terputus. Jika memilih `False`, file berada di runtime sementara dan dapat hilang saat runtime berakhir.

In [ ]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/TB_Guard_Training")
else:
    OUTPUT_DIR = Path("/content/tb_guard_training")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = OUTPUT_DIR / "mobilenetv3_cwt_best.pth"
LAST_PATH = OUTPUT_DIR / "mobilenetv3_cwt_last.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"
print("Output directory:", OUTPUT_DIR)

## 9. Training dengan progress bar dan resume

Progress bar menunjukkan kemajuan batch. Model terbaik disimpan berdasarkan F1 kelas TB pada validation set, dan checkpoint terakhir disimpan setiap epoch. Jika sesi putus, jalankan notebook lagi dari awal hingga cell ini; training akan mencoba resume dari checkpoint.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import copy

EPOCHS = 15
PATIENCE = 5
start_epoch = 0
best_val_f1 = -1.0
epochs_without_improvement = 0
history = []

if LAST_PATH.exists():
    try:
        ckpt = torch.load(LAST_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        if USE_AMP and ckpt.get("scaler_state_dict"):
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_val_f1 = float(ckpt["best_val_f1"])
        epochs_without_improvement = int(ckpt["epochs_without_improvement"])
        history = ckpt.get("history", [])
        print(f"Resuming from epoch {start_epoch + 1}; best val F1={best_val_f1:.4f}")
    except Exception as e:
        print("Could not resume; starting fresh:", str(e)[:250])

if start_epoch >= EPOCHS:
    print("Checkpoint has already reached EPOCHS. Increase EPOCHS if you want more training.")
else:
    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()
        running_loss = correct = total = 0

        bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} TRAIN")
        for images, labels in bar:
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()

            n = labels.size(0)
            running_loss += loss.item() * n
            correct += (logits.argmax(1) == labels).sum().item()
            total += n
            bar.set_postfix(loss=f"{running_loss/max(total,1):.4f}",
                            acc=f"{correct/max(total,1):.3f}")

        train_loss = running_loss / max(total, 1)
        train_acc = correct / max(total, 1)

        model.eval()
        val_true, val_pred = [], []
        val_loss_sum = val_total = 0
        with torch.no_grad():
            vbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} VALID")
            for images, labels in vbar:
                images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                    logits = model(images)
                    loss = criterion(logits, labels)
                val_loss_sum += loss.item() * labels.size(0)
                val_total += labels.size(0)
                val_true.extend(labels.cpu().numpy())
                val_pred.extend(logits.argmax(1).cpu().numpy())
                vbar.set_postfix(val_loss=f"{val_loss_sum/max(val_total,1):.4f}")

        val_loss = val_loss_sum / max(val_total, 1)
        val_acc = accuracy_score(val_true, val_pred)
        val_f1 = f1_score(val_true, val_pred, pos_label=TB_IDX, zero_division=0)
        scheduler.step(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_without_improvement = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "class_to_idx": train_ds.class_to_idx,
                "architecture": "mobilenet_v3_small",
                "image_size": IMAGE_SIZE,
                "best_val_f1_tb": best_val_f1,
                "epoch": epoch
            }, BEST_PATH)
        else:
            epochs_without_improvement += 1

        row = {"epoch": epoch+1, "train_loss": train_loss, "train_accuracy": train_acc,
               "val_loss": val_loss, "val_accuracy": val_acc, "val_f1_tb": val_f1,
               "lr": optimizer.param_groups[0]["lr"], "seconds": time.time()-t0}
        history.append(row)
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if USE_AMP else {},
            "best_val_f1": best_val_f1,
            "epochs_without_improvement": epochs_without_improvement,
            "history": history
        }, LAST_PATH)

        print(f"Epoch {epoch+1}/{EPOCHS} | train loss {train_loss:.4f} acc {train_acc:.4f} "
              f"| val loss {val_loss:.4f} acc {val_acc:.4f} F1_TB {val_f1:.4f} "
              f"| {time.time()-t0:.1f}s")

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping: validation F1 did not improve.")
            break

    print("Training finished. Best model:", BEST_PATH)
    print("Latest checkpoint:", LAST_PATH)

## 10. Evaluasi akhir pada test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

if not BEST_PATH.exists():
    raise FileNotFoundError("Best checkpoint belum ada. Jalankan training dahulu.")

best = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model_state_dict"])
model.eval()

y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    bar = tqdm(test_loader, desc="Final TEST evaluation")
    for images, labels in bar:
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
        probs = torch.softmax(logits, dim=1).float().cpu().numpy()
        y_true.extend(labels.numpy())
        y_pred.extend(probs.argmax(axis=1))
        y_prob.extend(probs[:, TB_IDX])

print(classification_report(y_true, y_pred, labels=list(range(len(train_ds.classes))),
                            target_names=train_ds.classes, digits=4, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(train_ds.classes))))
print("Confusion matrix (rows=actual, columns=predicted):\n", cm)
if len(set(y_true)) == 2:
    y_binary = [1 if y == TB_IDX else 0 for y in y_true]
    print("ROC-AUC:", round(roc_auc_score(y_binary, y_prob), 4))

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Test Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.xticks(range(len(train_ds.classes)), train_ds.classes, rotation=20)
plt.yticks(range(len(train_ds.classes)), train_ds.classes)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i,j]), ha="center", va="center")
plt.tight_layout(); plt.show()

## 11. Output dan batasan

File utama:
- `mobilenetv3_cwt_best.pth`: model terbaik berdasarkan validation F1.
- `mobilenetv3_cwt_last.pth`: checkpoint terakhir untuk resume.
- `training_history.csv`: metrik per epoch.

Jika menyimpan ke Drive, buka `MyDrive/TB_Guard_Training`. Model ini hanya mempelajari label yang tersedia pada dataset dan tidak boleh dipakai untuk diagnosis klinis. Audit ID pasien dari nama file adalah heuristik; verifikasi dokumentasi dataset untuk memastikan split benar-benar independen.